# ADB parser walkthrough

This notebook is the practical guide for parsing Asian Development Bank Excel workbooks in MARIO.

## What this notebook covers

- where to download the economic and air-emissions workbooks;
- when to pass a direct file path and when to pass a directory;
- how `year=` and `economies=` are used to disambiguate MRIO releases;
- how SRIO workbooks differ from MRIO workbooks;
- how `add_extensions=` imports the ADB air-emissions table;
- which parser warnings can appear and where to inspect them.

## Relevant source pages

- Official ADB MRIO and SRIO page: [ADB globalization portal](https://kidb.adb.org/globalization/current)
- Official ADB air-emissions extensions page: [ADB environmentally extended MRIOT](https://kidb.adb.org/globalization/adb_environmentally_extended_multiregional_inputoutput_tables)

MARIO does not download ADB workbooks automatically. The expected workflow is to download the `.xlsx` files manually from those pages and then point the parser to the local files.

## Main entry point

For normal user workflows, the public entry point is:

- `mario.parse_adb(...)`

ADB parsing currently supports only `IOT` tables, but the same public function works with:

- ADB `MRIO` workbooks;
- ADB `SRIO` workbooks;
- optional ADB air-emissions workbooks passed through `add_extensions=`.

## Key arguments

The key public arguments are:

- `path`: one ADB workbook or one directory containing multiple workbooks
- `table`: currently only `"IOT"` is supported
- `year`: used to disambiguate MRIO releases in one directory; mandatory for SRIO workbooks because one file contains multiple yearly sheets
- `economies`: used when a directory contains multiple MRIO variants for the same year, such as `62`, `72`, or `74` economies
- `add_extensions`: optional path to the ADB air-emissions workbook; this populates `E` and keeps `EY` zero-filled

## Direct file path vs directory path

Use a **direct workbook path** when you already know the exact file to parse. This is the simplest path and usually means you do not need `year=` or `economies=`.

Use a **directory path** when you keep several ADB workbooks together. In that case MARIO scans the directory, finds the candidate workbook files, and then uses `year=` and optionally `economies=` to pick the correct one.

In [ ]:
import mario

## Parse one explicit MRIO workbook

Use this when you want to point MARIO to one specific ADB MRIO workbook on disk.

In [ ]:
db_mrio = mario.parse_adb(
    path="/path/to/ADB_MRIO_2024_74.xlsx",
)

## Parse from a directory containing multiple MRIO releases

This is the important case when one directory contains several ADB MRIO files. Typical examples are:

- multiple years in the same folder;
- multiple release variants for the same year, such as `62`, `72`, or `74` economies.

In those cases, `year=` narrows the release year and `economies=` narrows the workbook variant.

In [ ]:
db_mrio = mario.parse_adb(
    path="/path/to/adb_directory",
    year=2024,
    economies=74,
)

If the directory contains only one MRIO workbook for the requested year, `year=` may be enough on its own.

In [ ]:
db_mrio = mario.parse_adb(
    path="/path/to/adb_directory",
    year=2023,
)

## Parse one SRIO workbook

ADB SRIO workbooks differ from MRIO workbooks in one key way: **one file contains multiple yearly sheets**. Because of that, `year=` is mandatory for SRIO parsing.

From the user side, the parser call stays simple: you still use `mario.parse_adb(...)`, but now `year=` selects the annual sheet inside the workbook.

In [ ]:
db_srio = mario.parse_adb(
    path="/path/to/CAN IOT 2000, 2007-2024.xlsx",
    year=2024,
)

## Add the ADB air-emissions extensions

When the matching environmental workbook is available locally, pass it through `add_extensions=`. MARIO imports the environmental extension matrix `E` from that workbook and keeps `EY` zero-filled.

The same argument works for both MRIO and SRIO economic tables.

In [ ]:
db_mrio_ext = mario.parse_adb(
    path="/path/to/ADB_MRIO_2024_74.xlsx",
    add_extensions="/path/to/2024 EE-MRIOT (Air Emissions).xlsx",
)

In [ ]:
db_srio_ext = mario.parse_adb(
    path="/path/to/CAN IOT 2000, 2007-2024.xlsx",
    year=2024,
    add_extensions="/path/to/2024 EE-MRIOT (Air Emissions).xlsx",
)

## Parser warnings you may see

When `add_extensions=` is used, MARIO can record parser warnings in the metadata history. The two relevant cases are:

- the emissions workbook year does not match the economic table year;
- the emissions workbook does not cover all regions present in the economic table.

The second case matters mainly for MRIO releases where the economic table contains regions not present in the emissions workbook. The parse still succeeds, but MARIO records the mismatch.

A simple way to inspect those parser notes is to check the metadata history after parsing:

In [ ]:
db_mrio_ext.meta_history

## Inspect the parsed database

Once parsed, the result is a standard MARIO database. At that point the normal exploration tools apply.

In [ ]:
db_mrio_ext